# Set-up

In [ ]:
import sys
sys.path.append('..')
from pathlib import Path
import numpy as np
import h5py
import jax
import jax.numpy as jnp
import yaml
import re
import collections
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
# plt.style.use('../flowrec/utils/ppt.mplstyle')
plt.style.use('../flowrec/utils/a4.mplstyle')

import flowrec.training_and_states as state_utils
import flowrec.data as data_utils
import flowrec.physics_and_derivatives as derivatives

from mpl_toolkits.axes_grid1 import ImageGrid, make_axes_locatable
from matplotlib import gridspec, patches
from scipy.interpolate import RBFInterpolator
from flowrec.utils import simulation
from flowrec import losses
from flowrec.utils.py_helper import slice_from_tuple
from flowrec.utils.system import set_gpu
from flowrec.utils.myplots import truegrey, create_custom_colormap, make_cax
cmap_trafficlight = create_custom_colormap('trafficlight')
cmap_trafficlight_pale = create_custom_colormap('trafficlight-pale')
from flowrec.utils import my_continuous_cmap, my_discrete_cmap

import os
# os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'
set_gpu(1,1)
# jax.config.update('jax_platform_name', 'cpu')

In [ ]:
from flowrec.training_and_states import restore_trainingstate, params_split, params_merge, generate_update_fn, TrainingState
from flowrec.data import unnormalise_group, normalise

In [ ]:
save_to_review = Path('./figs-3dkol-planes/review')

# Shared functions

## Stats of the repeats

- mean and standard deviation of relative error, physics loss 
- same for the time-averaged flow field
- list of folder names with corresponding loss values

In [ ]:
def get_summary_onecase(
        results_dir, 
        predict_only=False, 
        history_only=False, 
        verbose=0,
        noisy=False,
    ):
    '''Return information about a single case'''

    if history_only:
        _result_file = Path(results_dir,'results.h5')
        hist= {}
        with h5py.File(_result_file,'r') as hf:
            hist['train'] = np.array(hf.get("loss_train"))
            hist['val'] = np.array(hf.get("loss_val"))
            hist['div'] = np.array(hf.get("loss_div"))
            hist['momentum'] = np.array(hf.get("loss_momentum"))
            hist['sensors'] = np.array(hf.get("loss_sensors"))
            hist['true'] = np.array(hf.get('loss_train_true'))
            hist['val_true'] = np.array(hf.get('loss_val_true'))
        return pd.DataFrame(hist)
        
    with open(Path(results_dir,'config.yml'),'r') as f:
        cfg = yaml.load(f, Loader=yaml.UnsafeLoader)

    cfg.data_config.update({'data_dir':'.'+cfg.data_config.data_dir})
    datacfg = cfg.data_config
    traincfg = cfg.train_config

    data, datainfo = cfg.case.dataloader(datacfg)
    # print(data.keys())
    if datacfg.shuffle:
        idx_shuffle, idx_unshuffle = data_utils.shuffle_with_idx(np.sum(datacfg.train_test_split), rng = np.random.default_rng(datacfg.randseed))
    _keys_to_exclude = [
        'u_train_clean',
        'u_val_clean',
        'train_minmax',
        'val_minmax',
        'u_train',
        'u_val',
        'inn_train',
        'inn_val'
    ]
    
    prep_data, make_model = cfg.case.select_model(datacfg=datacfg, mdlcfg=cfg.model_config, traincfg=traincfg)
    data = prep_data(data, datainfo)
    inn_train = data['inn_train']
    u_train = data['u_train']
    # print(u_train[0].shape)
    mdl = make_model(cfg.model_config)
    state = restore_trainingstate(results_dir,'state')


    if Path(results_dir, 'frozen_params.npy').exists():
        params_frozen = restore_trainingstate(results_dir, 'frozen_params')
        # print('These layers are frozen: ')
        # for l in list(params_frozen):
        #     print("  ",l)
        full_params = params_merge(params_frozen, state.params)
        full_params = params_merge(state.params, params_frozen)
    else:
        full_params = state.params
    # print('All layers: ')
    # for l in list(full_params):
    #     print("  ",l)
    if verbose==1:
        param_count = sum(x.size for x in jax.tree_util.tree_leaves(full_params))
        print(f'Total number of parameters {param_count:,}')

    pred_train = []
    for _inn in inn_train:
        pred_train.append(
            mdl.predict(full_params, _inn)
        )
    pred_train = np.concatenate(pred_train, axis=0)

    observe_kwargs = {key: value for key, value in data.items() if key not in _keys_to_exclude}
    take_observation, insert_observation = cfg.case.observe(
        datacfg,
        example_pred_snapshot = data['u_train'][0][0,...],
        example_pin_snapshot = data['inn_train'][0][0,...],
        **observe_kwargs
    )

    _, train_minmax = take_observation(np.concatenate(data['u_train'],axis=0), init=True)
    _, val_minmax = take_observation(np.concatenate(data['u_val'],axis=0),init=True)
    observed_train = [take_observation(_u) for _u in data['u_train']]
    observed_val = [take_observation(_u) for _u in data['u_val']]
    data.update({
        'y_train':observed_train,
        'y_val':observed_val,
        'train_minmax':train_minmax,
        'val_minmax':val_minmax 
    })

    _empty_data = jnp.zeros_like(data['u_train'][0][[0],...]) - 50.0
    _empty_data = insert_observation(_empty_data, observed_train[0][[0],...])
    _empty_data = np.squeeze(_empty_data)
    has_values = _empty_data > -50.0

    if predict_only:
        return pred_train, has_values
    
    if noisy:
        ref = data['u_train_clean']
        if isinstance(ref,list):
            ref = np.concatenate(ref, 0)
        if isinstance(u_train,list):
            ref_noisy = np.concatenate(u_train,0)
        return pred_train, has_values, ref, datainfo, data['forcing'], ref_noisy
    else:
        if isinstance(u_train,list):
            ref = np.concatenate(u_train,0)
        return pred_train, has_values, ref, datainfo, data['forcing']



In [ ]:
def get_losses_repeats(repeat_dir, prefix, history=False):
    folders = [f for f in repeat_dir.iterdir() if f.is_dir()]
    pattern = re.compile(f'^{prefix}')
    folders = [f for f in folders if re.search(pattern, f.name)]
    print(len(folders),folders)

    l = collections.defaultdict(list)
    for i in range(len(folders)):
        print(f'    run{i}')
        if i == 0:
            pred_train, has_values, ref, datainfo, forcing = get_summary_onecase(folders[i])
        else:
            pred_train, has_values = get_summary_onecase(folders[i], predict_only=True)

        l['rel-l2'].append(float(
            losses.relative_error(pred_train, ref)
        ))
        l['l2'].append(float(
            losses.mse(pred_train,ref)
        ))
        observed_ref = ref[:,has_values]
        observed_pred = pred_train[:,has_values]
        l['sensor'].append(float(
            losses.mse(observed_pred, observed_ref)
        ))
        with jax.default_device(jax.devices('cpu')[0]):
            l['momentum'].append(float(
                losses.momentum_loss(pred_train,datainfo,forcing=forcing)
            ))
            l['div'].append(float(
                losses.divergence(pred_train[...,:-1], datainfo)
            ))
    l['rel-l2'].append(np.nan)
    l['l2'].append(np.nan)
    l['sensor'].append(np.nan)
    with jax.default_device(jax.devices('cpu')[0]):
        l['momentum'].append(float(
            losses.momentum_loss(ref, datainfo, forcing=forcing)
        ))
        l['div'].append(float(
            losses.divergence(ref[...,:-1], datainfo, forcing=forcing)
        ))
    indexes = [f.name for f in folders]
    indexes.append('ref')
    df = pd.DataFrame(l, index=indexes)
    df['physics'] = df['momentum'] + df['div']
    df['total'] = df['physics'] + df['sensor']
    df.sort_values('total', inplace=True)
    return df, ref 

In [ ]:
def get_surface_of_box_index(data):
    x,y,z = np.indices(data.shape)
    # 6 faces x=0,-1, y=0,-1, z=0,-1
    f1 = x==0
    f2 = x==x.max()
    f3 = y==0
    f4 = y==y.max()
    f5 = z==0
    f6 = z==z.max()
    faces = [f1,f2,f3,f4,f5,f6]
    x1 = []
    y1 = []
    z1 = []
    surface = []
    for f in faces:
        x1.extend(list(x[f]))
        y1.extend(list(y[f]))
        z1.extend(list(z[f]))
        surface.extend(list(data[f]))
    return (x1,y1,z1), surface

# Plots

In [ ]:
def get_data_convergence(*filenames):
    sum_sets = []
    nt_sets = []
    tke_sets = []
    kgrid_magnitude_int = None

    for f in filenames:
        print(f)
        with h5py.File(f,'r') as hf:
            u_p = np.array(hf.get('state'))
            ndim = int(hf.get('ndim')[()])
            dt = float(hf.get('dt')[()])
            re = float(hf.get('re')[()])
        ufluc = u_p[...,:-1]-np.mean(u_p[...,:-1],axis=0)
        dx = [2*np.pi/u_p.shape[1]]*ndim
        d = [dt]
        d.extend(dx)
        datainfo = data_utils.DataMetadata(
            re=re,
            discretisation=d,
            axis_index=list(range(ndim+1)),
            problem_2d= (ndim==2)
        ).to_named_tuple()

        if kgrid_magnitude_int is None:
            fftfreq = []
            dk = 2*np.pi/np.array(dx)
            for i in range(ndim):
                _k = np.fft.fftfreq(ufluc.shape[1:-1][i])*dk[i]
                fftfreq.append(_k)
            
            kgrid = np.meshgrid(*fftfreq, indexing='ij')
            kgrid = np.array(kgrid)
            kgrid_magnitude = np.sqrt(np.einsum('n... -> ...', kgrid**2))
            kgrid_magnitude_int = kgrid_magnitude.astype('int')
        spectrum, kbins = derivatives.get_tke(ufluc, datainfo,kgrid_magnitude=kgrid_magnitude_int)

        tke_sets.append(spectrum)
        nt_sets.append(u_p.shape[0])
        sum_sets.append(np.sum(u_p,axis=0))
    sum_sets = np.array(sum_sets)
    
    sum_all = np.sum(sum_sets,axis=0)
    nt_all = np.sum(nt_sets)

    mean_all = sum_all/nt_all
    tke_all = np.sum(tke_sets,axis=0)

    data_dict = {'mean_all': mean_all, 'tke_all':tke_all, 'kbins':kbins}
    return data_dict

def plot_data_convergence(data_dict, figsize):
    mean_all, tke_all, kbins = data_dict['mean_all'], data_dict['tke_all'], data_dict['kbins']
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 6, width_ratios=[15,1,15,1,6,15], wspace=0.3, left=0.03, right=0.98, hspace=0.5)


    _pad = fig.add_subplot(gs[4])
    _pad.scatter([0,1],[0,1],color='w')
    _pad.axis('off')
    _pad = fig.add_subplot(gs[10])
    _pad.scatter([0,1],[0,1],color='w')
    _pad.axis('off')

    ## plot mean

    component_labels = ['$u_1$','$u_2$','$u_3$','$p$',]
    axes = {}
    for i, iax in enumerate([0,2,6,8]):
        ax = fig.add_subplot(gs[iax], projection='3d')
        ax.set_xlim(0.0, mean_all.shape[0])
        ax.set_ylim(0.0, mean_all.shape[1])
        ax.set_zlim(0.0, mean_all.shape[2])
        ax.set_xticks([2, mean_all.shape[0]-2], ['0', '$2\pi$'])
        ax.set_yticks([2, mean_all.shape[1]-2], ['0', '$2\pi$'])
        ax.set_zticks([2, mean_all.shape[2]-2], ['0', '$2\pi$'])
        ax.view_init(elev=30, azim=120)
        ax.tick_params(pad=0.2, labelsize='x-small')
        ax.set_title(component_labels[i],fontsize='medium')
        vmin, vmax = mean_all[...,i].min(), mean_all[...,i].max()
        (x1,y1,z1), plt_data = get_surface_of_box_index(mean_all[...,i])
        sc = ax.scatter(x1, y1, z1, c=plt_data, marker='s', s=2, vmin=vmin, vmax=vmax, edgecolors=None)

        # colorbar
        icax = iax + 1
        cax = fig.add_subplot(gs[icax])
        cbar = fig.colorbar(sc, cax=cax)
        cbar.ax.tick_params(labelsize='x-small')
        axes[f'scatter{i}'] = ax

    ## plot physics
    ax = fig.add_subplot(gs[5])
    ax.loglog(kbins, tke_all, color=my_discrete_cmap(0))
    ax.set_ylabel('TKE', fontsize='small')
    ax.set_xlabel('Wavenumber', fontsize='small')
    ax.set_xlim(0, ((32**2)*3)**0.5)
    ax.tick_params(labelsize='x-small')
    ax.grid()
    axes['tke'] = ax

    # plot forcing
    x = np.linspace(0, 2*np.pi, mean_all.shape[0])
    ax = fig.add_subplot(gs[11])
    ax.plot(x, np.mean(mean_all[...,0], axis=(1,2)), label='$x_1$', color=my_discrete_cmap(2))
    ax.plot(x, np.mean(mean_all[...,0], axis=(0,2)), label='$x_2$', color=my_discrete_cmap(0))
    ax.plot(x, np.mean(mean_all[...,0], axis=(0,1)), label='$x_3$', color=my_discrete_cmap(1))
    ax.plot(x, np.sin(4*x), label='Forcing', color='k', linestyle='dashed')
    ax.set(xlim=[0.0,2*np.pi], ylim=[-1.05,1.4],xticks=[0.0,2*np.pi],xticklabels=[0.0, '$2\pi$'])
    ax.tick_params(labelsize='x-small')
    ax.set_ylabel('$u_1$', fontsize='medium')
    ax.legend(ncol=2, fontsize='xx-small', loc='upper center', bbox_to_anchor=(0.5,1.15))
    axes['forcing'] = ax

    return fig, axes

In [ ]:
from scipy.ndimage import gaussian_filter
def subplot_vort_strength(ax3d, snapshot, lower_threshold, sigma=0, s=1, higher_threshold=None, cmap='gray', alpha=1, markeredgecolor='none', vmin=None, vmax=None):

    snapshot_smooth = gaussian_filter(snapshot, sigma=sigma)
    idx = lower_threshold < snapshot_smooth
    if higher_threshold is not None:
        idx_higher = snapshot_smooth < higher_threshold
        idx = idx & idx_higher

    x, y, z = np.indices(snapshot.shape)
    x1 = x[idx]
    y1 = y[idx]
    z1 = z[idx] 
    values = snapshot[idx]

    # Flatten the arrays to use in scatter plot
    x1, y1, z1, values = x1.flatten(), y1.flatten(), z1.flatten(), values.flatten()

    
    ax3d.view_init(elev=30, azim=120)
    sc = ax3d.scatter(x1, y1, z1, c=values, marker='s', s=s, cmap=cmap, alpha=alpha, edgecolor=markeredgecolor, vmin=vmin, vmax=vmax)

    ax3d.set_xlim(0.0, snapshot.shape[0])
    ax3d.set_ylim(0.0, snapshot.shape[1])
    ax3d.set_zlim(0.0, snapshot.shape[2])
    ax3d.set_xticks([2, snapshot.shape[0]-2], ['0', '$2\pi$'])
    ax3d.set_yticks([2, snapshot.shape[1]-2], ['0', '$2\pi$'])
    ax3d.set_zticks([2, snapshot.shape[2]-2], ['0', '$2\pi$'])
    ax3d.tick_params(pad=0.1, labelsize='x-small')
    ax3d.set_xlabel("$x_1$",fontsize='small')
    ax3d.set_ylabel("$x_2$",fontsize='small')
    ax3d.set_zlabel("$x_3$",fontsize='small')

    return ax3d, sc

In [ ]:
def plot_observations(snapshot, has_values, figsize=(5,5), s=1, highlight_inn=False, alpha=0.5, zorder=1):
    x, y, z, u = np.indices(snapshot.shape)
    x1 = x[has_values]
    y1 = y[has_values]
    z1 = z[has_values] 
    u1 = u[has_values]
    values = snapshot[has_values]

    # Flatten the arrays to use in scatter plot
    x1, y1, z1, u1, values = x1.flatten(), y1.flatten(), z1.flatten(), u1.flatten(), values.flatten()

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 2, wspace=0.7, right=0.85, bottom=0.05, left=0.05, top=0.95) 

    # Scatter plot where color is based on cube values
    axes = []
    labels = ['$u_1$','$u_2$','$u_3$','$p$']
    for i in range(4):
        ax = fig.add_subplot(gs[i], projection='3d',computed_zorder=False)
        axes.append(ax)
        ax.view_init(elev=30, azim=120)
        ui = u1 == i
        sc = ax.scatter(x1[ui], y1[ui], z1[ui], c=values[ui], marker='.', s=s, alpha=alpha, zorder=2)
        ax.set_xlim(0.0, snapshot.shape[0])
        ax.set_ylim(0.0, snapshot.shape[1])
        ax.set_zlim(0.0, snapshot.shape[2])
        ax.set_xticks([2, snapshot.shape[0]-2], ['0', '$2\pi$'])
        ax.set_yticks([2, snapshot.shape[1]-2], ['0', '$2\pi$'])
        ax.set_zticks([2, snapshot.shape[2]-2], ['0', '$2\pi$'])
        ax.tick_params(pad=0.1, labelsize='x-small')
        ax.set_xlabel("$x_1$",fontsize='small')
        ax.set_ylabel("$x_2$",fontsize='small')
        ax.set_zlabel("$x_3$",fontsize='small')
        ax.set_title(labels[i], fontsize='medium')
    
    num_observed = np.count_nonzero(has_values)
    num_total = snapshot.size
    print(f'{num_observed} measurements, {num_observed/num_total:%} of the total {num_total} grid points.')

    if highlight_inn:
        inn_loc = has_values[...,-1] != has_values[...,0]
        inn_loc = inn_loc & has_values[...,-1]
        print(np.count_nonzero(inn_loc))
        inns = snapshot[inn_loc,-1] 
        x2 = x[inn_loc,-1]
        y2 = y[inn_loc,-1]
        z2 = z[inn_loc,-1] 
        x2, y2, z2, inns = x2.flatten(), y2.flatten(), z2.flatten(), inns.flatten()
        axes[-1].scatter(x2, y2, z2, c=inns, marker='o', s=s, cmap='Reds', zorder=zorder)

    return fig, axes


# def plot_inn_sensor_loc(snapshot, has_values, figsize=(3,3), s=1):
#     inn_loc = has_values[...,-1] != has_values[...,0]

In [ ]:
def plot_volumes(ref_snapshot, pred_snapshot, figsize=(6.5,2.5), label=None, titles=['',''], s=1):
    """snapshot of a single component"""
    # x, y, z = np.indices(ref_snapshot.shape)
    # # Flatten the arrays to use in scatter plot
    # x1, y1, z1 = x.flatten(), y.flatten(), z.flatten()
    vmin, vmax = ref_snapshot.min(), ref_snapshot.max()

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(1, 3, width_ratios=[12, 12, 1], wspace=0.5, left=0.05, right=0.88) 
    cax = fig.add_subplot(gs[-1])

    # Scatter plot where color is based on cube values
    axes = []
    for i in range(2):
        ax = fig.add_subplot(gs[i], projection='3d')
        ax.view_init(elev=30, azim=120)
        ax.set_xlim(0.0, ref_snapshot.shape[0])
        ax.set_ylim(0.0, ref_snapshot.shape[1])
        ax.set_zlim(0.0, ref_snapshot.shape[2])
        ax.set_xticks([2, ref_snapshot.shape[0]-2], ['0', '$2\pi$'])
        ax.set_yticks([2, ref_snapshot.shape[1]-2], ['0', '$2\pi$'])
        ax.set_zticks([2, ref_snapshot.shape[2]-2], ['0', '$2\pi$'])
        ax.tick_params(pad=0.2, labelsize='x-small')
        ax.set_title(titles[i],fontsize='medium')
        axes.append(ax)
    (x1,y1,z1), plt_ref = get_surface_of_box_index(ref_snapshot)
    sc0 = axes[0].scatter(x1, y1, z1, c=plt_ref, marker='s', s=s, vmin=vmin, vmax=vmax, edgecolors=None)
    cbar = fig.colorbar(sc0, cax=cax, label=label)
    cbar.ax.tick_params(labelsize='x-small') 
    (x1,y1,z1), plt_pred = get_surface_of_box_index(pred_snapshot)
    sc1 = axes[1].scatter(x1, y1, z1, c=plt_pred, marker='s', s=s, vmin=vmin, vmax=vmax, edgecolors=None)
    fig.text(0.1,0.1,'$x_1$',rotation=345)
    fig.text(0.33,0.17,'$x_2$',rotation=55)
    fig.text(0.5,0.1,'$x_1$',rotation=345)
    fig.text(0.73,0.17,'$x_2$',rotation=55)
    fig.text(0.80,0.5,'$x_3$')
    return fig, axes

In [ ]:
def plot_pdf(pred, ref, figsize=(6.5,2.5)):
    labels = ['$u\'_1$', '$u\'_2$', '$u\'_3$', '$p\'$']
    with jax.default_device(jax.devices("cpu")[0]):
        fig, axes = plt.subplots(1,4,figsize=figsize)
        for i,ax in enumerate(axes):
            counts,bins = np.histogram(ref[...,i].flatten()-np.mean(ref[...,i].flatten()), density=True, bins='auto')
            l0 = ax.stairs(counts,bins,label='Reference',linewidth=2,color=truegrey)
            counts,bins = np.histogram(pred[...,i].flatten()-np.mean(pred[...,i].flatten()), density=True, bins='auto')
            l1 = ax.stairs(counts,bins,label='Reconstructed',linewidth=1, color=my_discrete_cmap(0))
            ax.tick_params(labelsize='x-small')
            ax.set_xlabel(labels[i])
            l0.set_rasterized(True)
            l1.set_rasterized(True)
        axes[0].set_ylabel('Probability density')
        handles, labels = axes[0].get_legend_handles_labels()
        fig.subplots_adjust(top=0.8, wspace=0.4, bottom=0.2, right=0.95)
        fig.legend(handles=handles, ncol=2, loc='upper center', bbox_to_anchor=(0.5,1.0))
    return fig, axes

In [ ]:
def plot_tke(ref, pred_list:list, pred_label_list:list, datainfo, figsize=(3.5,2.5), linewidth=2, wavenumber=32, fgen=False, fnyquist=False, log=True, spectrum=None):
    if spectrum:
        spectrum_true = spectrum['spectrum_true']
        spectrum_pred_list = spectrum['spectrum_pred_list']
        kbins = spectrum['kbins']
        kbins_list = spectrum['kbins_list']
    else:
        spectrum_true, kbins = derivatives.get_tke(ref[...,:-1]-np.mean(ref[...,:-1],axis=0), datainfo, domain_size=2*np.pi*np.array(ref.shape[1:-1])/64.)
        spectrum_pred_list, kbins_list = [], []
        for pred in pred_list:
            spectrum_pred, _kbins = derivatives.get_tke(pred[...,:-1]-np.mean(pred[...,:-1],axis=0), datainfo, domain_size=2*np.pi*np.array(pred.shape[1:-1])/64.)
            spectrum_pred_list.append(spectrum_pred)
            kbins_list.append(_kbins)
    f_generated = int(np.sqrt((ref.ndim-2)*wavenumber**2))
    gridsize = np.array(ref.shape[1:-1]) / 2.0
    f_nyquist = int(np.sqrt(np.sum(gridsize**2)))
    xmax = max(f_generated, f_nyquist)

    fig, ax = plt.subplots(1,1, figsize=figsize)
    fig.subplots_adjust(top=0.95, bottom=0.19, right=0.95, left=0.25)

    ax.plot(kbins, spectrum_true, label='Reference', color=truegrey, linewidth=linewidth+1)
    for i, (spectrum_pred, label) in enumerate(zip(spectrum_pred_list, pred_label_list)):
        ax.plot(kbins_list[i], spectrum_pred, label=label, color=my_discrete_cmap(i), linewidth=linewidth)
    if log:
        ax.set_yscale('log')
    if fgen:
        ax.vlines(f_generated, spectrum_true.min(), spectrum_true.max(),colors='k',linestyles='dotted', label='$k_{gen}$')
    if fnyquist:
        ax.vlines(f_nyquist, spectrum_true.min(), spectrum_true.max(),colors='k',linestyles='dashed', label='$k_{Nyquist}$')
    
    # ax.set_xlim(0, xmax+3)
    ax.grid()
    ax.legend()
    ax.set_xlabel('Wavenumber',fontsize='medium')
    ax.set_ylabel('Turbulent Kinetic Energy',fontsize='medium')
    # handles, labels = ax.get_legend_handles_labels()
    # fig.legend(handles=handles, ncol=2, loc='upper center', bbox_to_anchor=(0.5,1.0))
    spectrum = {
        'spectrum_true': spectrum_true,
        'spectrum_pred_list': spectrum_pred_list,
        'kbins': kbins,
        'kbins_list': kbins_list
    }
    return fig, ax, spectrum

In [ ]:
def plot_compare_networks(ref_snapshot_list, pred1_list, pred2_list, figsize=(6.5,3), labels=['',''], s=1):
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2,4, width_ratios=[12,12,12,1], wspace=0.5, left=0.05, right=0.88, hspace=0.5)
    print(gs)
    cax0 = fig.add_subplot(gs[0,-1])
    cax1 = fig.add_subplot(gs[1,-1])
    cax = [cax0, cax1]

    # Scatter plot where color is based on cube values
    axes = []
    for i, plt_data in enumerate(zip(ref_snapshot_list, pred1_list, pred2_list)):
        _, _plt_ref = get_surface_of_box_index(plt_data[0])
        # vmin, vmax = min(_plt_ref), max(_plt_ref)
        _std = np.std(_plt_ref)
        _mean = np.mean(_plt_ref)
        vmin = _mean - 3*_std
        vmax = _mean + 3*_std
        for j, data in enumerate(plt_data):
            ax = fig.add_subplot(gs[i,j], projection='3d')
            # print(i,j)
            ax.view_init(elev=30, azim=120)
            (x1,y1,z1), surface = get_surface_of_box_index(data)
            sc = ax.scatter(x1, y1, z1, c=surface, marker='s', s=s, vmin=vmin, vmax=vmax, edgecolors=None)
            axes.append(ax)
        cbar = fig.colorbar(sc, cax=cax[i], label=labels[i])
        cbar.ax.tick_params(labelsize='x-small') 

    for ax in axes:
        ax.set_xlim(0.0, ref_snapshot_list[0].shape[0])
        ax.set_ylim(0.0, ref_snapshot_list[0].shape[1])
        ax.set_zlim(0.0, ref_snapshot_list[0].shape[2])
        ax.set_xticks([2, ref_snapshot_list[0].shape[0]-2], ['0', '$2\pi$'])
        ax.set_yticks([2, ref_snapshot_list[0].shape[1]-2], ['0', '$2\pi$'])
        ax.set_zticks([2, ref_snapshot_list[0].shape[2]-2], ['0', '$2\pi$'])
        ax.tick_params(pad=0.2, labelsize='x-small')



    return fig, axes

In [ ]:
def plot_error_v_distance(*err, dz=2*np.pi/64, measured_iz=[], labels=[]):
    fig, ax = plt.subplots(1,1,figsize=(3.8,3))
    for i, (e,l) in enumerate(zip(err,labels)):
        plotz = np.arange(len(e))*dz
        ax.plot(plotz, e, label=l, c=my_discrete_cmap(i))
    ax.vlines([a*dz for a in measured_iz], ymin=0.0, ymax=0.5, colors='k', linestyles='dashed', label='Measured\nplanes')
    ticks = [0]
    ticks.extend(measured_iz)
    ticks.extend([63])
    ax.set(
        xlabel='$x_3$', 
        ylabel="Root Mean Square Error", 
        xticks=np.array(ticks)*dz, 
        xticklabels=[*[f"{a*dz/np.pi:.1f}$\pi$" for a in ticks[:-1]], "2$\pi$"], 
        xlim=(0,plotz[-1])
    )
    return fig, ax

# Extend 2D method

In [ ]:
save_to_2dmethod = './figs-3dkol-2dmethod/'
results_dir_2dmethod = Path('../local_results/3dkol/repeats_methods')
losses_2dmethod, ref = get_losses_repeats(results_dir_2dmethod, '2dmethod')

In [ ]:
print(losses_2dmethod.iloc[:-1,:].mean(), '\n', losses_2dmethod.iloc[:-1,:].std())
folders_2dmethod = list(losses_2dmethod.index[:-1])
print("Cases sorted by total loss ", folders_2dmethod)
losses_2dmethod

In [ ]:
pred_2dmethod, sensors_2dmethod, ref, datainfo, forcing = get_summary_onecase(Path(results_dir_2dmethod,folders_2dmethod[0]))

In [ ]:
with jax.default_device(jax.devices('cpu')[0]):
    vort = []
    vort_2dmethod = []
    batch = 500
    for i in range(ref.shape[0]//batch):
        _vort = derivatives.vorticity(ref[i*500:(i+1)*500,...,:-1], datainfo)
        vort.append(np.einsum('vt... -> t...v', _vort))
        _vort_2dmethod = derivatives.vorticity(pred_2dmethod[i*500:(i+1)*500,...,:-1], datainfo)
        vort_2dmethod.append(np.einsum('vt... -> t...v', _vort_2dmethod))

    vort = np.concatenate(vort, axis=0)
    vort_2dmethod = np.concatenate(vort_2dmethod, axis=0)

    v_abs = np.sqrt(np.einsum('t...v -> t...', vort**2))
    v_2dmethod_abs = np.sqrt(np.einsum('t...v -> t...', vort_2dmethod**2))

In [ ]:
v_abs_mean = v_abs.mean()
v_abs_std = v_abs.mean(axis=0).std()

In [ ]:
## Plot mean vorticity isosurface
fig = plt.figure(figsize=(6.5,2.5))
gs = fig.add_gridspec(1, 3, wspace=0.4, width_ratios=[15,15,1], left=0.02, right=0.87, bottom=0.2, top=0.95)
ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs.mean(axis=0), lower_threshold=v_abs_mean+1*v_abs_std, s=5, sigma=0, cmap='gray', alpha=0.7)
ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_2dmethod_abs.mean(axis=0), lower_threshold=v_abs_mean+1*v_abs_std, s=5, sigma=0, cmap='gray', alpha=0.7)
cax = fig.add_subplot(gs[2])
cbar = fig.colorbar(scref, cax=cax, label='$\overline{|\omega|}$')
# plt.show()
# fig.savefig(save_to_review / '2dmethod-volume-vabs.png')


In [ ]:
fig1, axes = plot_observations(ref[0,...], has_values=sensors_2dmethod, figsize=(3.8,3.8), highlight_inn=True, alpha=0.2)
# fig1.savefig(save_to_2dmethod+'sensors')

In [ ]:
# for this case, plot the cross plane
# grid1a x-y planes at two z locations (ref, pred, error) vorticity, grid1b ... pressure.
# grid2a y-z planes at two x locations (ref, pred, error) vorticity, grid2b ... pressure.

index_plot = [20,40]
plt_t = 100

fig = plt.figure(figsize=(6,6))
grid1a = ImageGrid(fig, (0.08, 0.54, 0.42, 0.38), (3,2), cbar_mode='edge', share_all=True)
grid1b = ImageGrid(fig, (0.55, 0.54, 0.42, 0.38), (3,2), cbar_mode='edge', share_all=True)
grid2a = ImageGrid(fig, (0.08, 0.06, 0.42, 0.38), (3,2), cbar_mode='edge', share_all=True)
grid2b = ImageGrid(fig, (0.55, 0.06, 0.42, 0.38), (3,2), cbar_mode='edge', share_all=True)

fig.text(0.03,0.57,'Error',rotation=90)
fig.text(0.03,0.69,'Recons.',rotation=90)
fig.text(0.03,0.85,'Ref.',rotation=90)
fig.text(0.03,0.57-0.48,'Error',rotation=90)
fig.text(0.03,0.69-0.48,'Recons.',rotation=90)
fig.text(0.03,0.85-0.48,'Ref.',rotation=90)
fig.text(0.32,0.97, f'Snapshots shown at t={plt_t*datainfo.dx:.2f}', fontsize='large')

vplotz = v_abs[plt_t, :, :, index_plot] # advance index goes to the front
pplotz = ref[plt_t, :, :, index_plot, -1]
vplot_predz = v_2dmethod_abs[plt_t, :, :, index_plot]
pplot_predz = pred_2dmethod[plt_t, :, :, index_plot, -1]
vplotx = v_abs[plt_t, index_plot, :, :]
pplotx = ref[plt_t, index_plot, :, :, -1]
vplot_predx = v_2dmethod_abs[plt_t, index_plot, :, :]
pplot_predx = pred_2dmethod[plt_t, index_plot, :, :, -1]
vmax = [vplotz.max(),vplotx.max()]
vmin = [vplotz.min(),vplotx.min()]
pmax = [pplotz.max(),pplotx.max()]
pmin = [pplotz.min(),pplotx.min()]
num_snapshots = len(index_plot)
for i in range(num_snapshots):
    iref, ipred, ierr = i, i+num_snapshots, i+2*num_snapshots

    imvref = grid1a.axes_all[iref].imshow(vplotz[i,...].T, vmin=vmin[0], vmax=vmax[0])
    grid1a.cbar_axes[0].colorbar(imvref, label='$|v|$')
    imvpred = grid1a.axes_all[ipred].imshow(vplot_predz[i,...].T, vmin=vmin[0], vmax=vmax[0])
    grid1a.cbar_axes[1].colorbar(imvpred, label='$|v|$')
    imverr = grid1a.axes_all[ierr].imshow(np.abs(vplotz[i,...] - vplot_predz[i,...]).T)
    grid1a.cbar_axes[2].colorbar(imverr, label='abs. err. $|v|$')
    
    impref = grid1b.axes_all[iref].imshow(pplotz[i,...].T, vmin=pmin[0], vmax=pmax[0])
    grid1b.cbar_axes[0].colorbar(impref, label='$p$')
    imppred = grid1b.axes_all[ipred].imshow(pplot_predz[i,...].T, vmin=pmin[0], vmax=pmax[0])
    grid1b.cbar_axes[1].colorbar(imppred, label='$p$')
    imperr = grid1b.axes_all[ierr].imshow(np.abs(pplotz[i,...] - pplot_predz[i,...]).T)
    grid1b.cbar_axes[2].colorbar(imperr, label=' abs. err. $p$')

    imvref = grid2a.axes_all[iref].imshow(vplotx[i,...].T, vmin=vmin[1], vmax=vmax[1])
    grid2a.cbar_axes[0].colorbar(imvref, label='$|v|$')
    imvpred = grid2a.axes_all[ipred].imshow(vplot_predx[i,...].T, vmin=vmin[1], vmax=vmax[1])
    grid2a.cbar_axes[1].colorbar(imvpred, label='$|v|$')
    imverr = grid2a.axes_all[ierr].imshow(np.abs(vplotx[i,...] - vplot_predx[i,...]).T)
    grid2a.cbar_axes[2].colorbar(imverr, label='abs. err. $|v|$')
    
    impref = grid2b.axes_all[iref].imshow(pplotx[i,...].T, vmin=pmin[1], vmax=pmax[1])
    grid2b.cbar_axes[0].colorbar(impref, label='$p$')
    imppred = grid2b.axes_all[ipred].imshow(pplot_predx[i,...].T, vmin=pmin[1], vmax=pmax[1])
    grid2b.cbar_axes[1].colorbar(imppred, label='$p$')
    imperr = grid2b.axes_all[ierr].imshow(np.abs(pplotx[i,...] - pplot_predx[i,...]).T)
    grid2b.cbar_axes[2].colorbar(imperr, label='abs. err. $p$')

for grid in [grid1a, grid1b]:
    grid[0].set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[2,58], xticklabels=['0', '$2\pi$'])
    grid.axes_all[0].set_title(f'$x_3$={index_plot[0]*datainfo.dz:.2f}', fontsize='medium')
    grid.axes_all[1].set_title(f'$x_3$={index_plot[1]*datainfo.dz:.2f}', fontsize='medium')
    for ax in grid.axes_all:
        ax.tick_params(length=0, labelsize='small')
        ax.set_xlabel('$x_1$',labelpad=0)
        ax.set_ylabel('$x_2$', labelpad=0)
for grid in [grid2a, grid2b]:
    grid[0].set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[2,58], xticklabels=['0', '$2\pi$'])
    grid.axes_all[0].set_title(f'$x_1$={index_plot[0]*datainfo.dx:.2f}', fontsize='medium')
    grid.axes_all[1].set_title(f'$x_1$={index_plot[1]*datainfo.dx:.2f}', fontsize='medium')
    for ax in grid.axes_all:
        ax.tick_params(length=0, labelsize='small')
        ax.set_xlabel('$x_2$',labelpad=0.0)
        ax.set_ylabel('$x_3$', labelpad=0.0)

# fig.savefig(save_to_2dmethod+'snapshots_slices')

In [ ]:
fig2,axes = plot_pdf(pred_2dmethod, ref, figsize=(7.5,2.8))
# fig2.savefig(save_to_2dmethod+'probability')

In [ ]:
fig3, axes = plot_volumes(v_abs[plt_t,...], v_2dmethod_abs[plt_t,...], label='$|v|$', titles=['Ref.', 'Recons.'], figsize=(3.5,2), s=2)
fig4, axes = plot_volumes(np.mean(v_abs,axis=0), np.mean(v_2dmethod_abs,axis=0), label='$|v|$', titles=['Ref.', 'Recons.'], figsize=(3.5,2), s=2)
# fig3.savefig(save_to_2dmethod + 'volume-vort-snapshot', bbox_inches='tight')
# fig4.savefig(save_to_2dmethod + 'volume-vort-mean', bbox_inches='tight')

In [ ]:
fig5, axes = plot_volumes(ref[plt_t,...,-1], pred_2dmethod[plt_t,...,-1], label='$p$', titles=['Ref.', 'Recons.'], figsize=(3.5,2), s=2)
fig6, axes = plot_volumes(np.mean(ref[...,-1],axis=0), np.mean(pred_2dmethod[...,-1],axis=0), label='$p$', titles=['Ref.', 'Recons.'], figsize=(3.5,2), s=2)
# fig5.savefig(save_to_2dmethod + 'volume-pressure-snapshot', bbox_inches='tight')
# fig6.savefig(save_to_2dmethod + 'volume-pressure-mean', bbox_inches='tight')

In [ ]:
fig7, ax, spectrum_2dmethod = plot_tke(ref, [pred_2dmethod], ['Reconstructed'], datainfo, log=True, spectrum=spectrum_2dmethod)
ax.set_xscale('log')
plt.show()
# fig7.savefig(save_to_2dmethod + 'tke')
# fig7.savefig(save_to_2dmethod + 'tke-linear')

In [ ]:
## 3d plots
# from plotly.subplots import make_subplots
# x, y, z = np.meshgrid(np.linspace(-1,1,20), np.linspace(-1,1,20), np.linspace(-1,1,20))
# values = np.sin(np.pi * x) * np.sin(np.pi * y) * np.sin(np.pi * z)

# # Create subplots with 2x2 layout and 3D support
# fig = make_subplots(
#     rows=2, cols=2,
#     specs=[[{'type': 'scene'}, {'type': 'scene'}],
#            [{'type': 'scene'}, {'type': 'scene'}]]
# )

# # Add 3D volume plots to each subplot
# for i in range(2):
#     for j in range(2):
#         scene_id = f'scene{i*2 + j + 1}'  # scene1, scene2, ...
#         fig.add_trace(
#             go.Volume(
#                 x=x.flatten(),
#                 y=y.flatten(),
#                 z=z.flatten(),
#                 value=values.flatten(),
#                 opacity=0.1,
#                 surface_count=15,
#                 showscale=False
#             ),
#             row=i+1, col=j+1
#         )

# # Optional: Customize each subplot layout
# fig.update_layout(
#     height=800,
#     width=800,
#     margin=dict(l=10, r=10, t=30, b=10),
#     scene=dict(aspectmode='cube'),
#     scene2=dict(aspectmode='cube'),
#     scene3=dict(aspectmode='cube'),
#     scene4=dict(aspectmode='cube')
# )

# fig.show()


# Reconstruct from slices

In [ ]:
plt_t_planes = 800

In [ ]:
save_to_planes = Path('./figs-3dkol-planes/')
results_dir_planes_notshare = Path('../local_results/3dkol/repeats_planes_notshare')
results_dir_planes_share = Path('../local_results/3dkol/repeats_planes_share')

In [ ]:
losses_planes2_notshare, ref = get_losses_repeats(results_dir_planes_notshare, 'notshare-plane2')
losses_planes2_share, _ = get_losses_repeats(results_dir_planes_share, 'share-plane2')

In [ ]:
print(losses_planes2_share.mean(), '\n', losses_planes2_share.std())
print('')
print(losses_planes2_notshare.mean(), '\n', losses_planes2_notshare.std())

In [ ]:
pred_planes2_share, sensors_planes2_share, ref, datainfo, forcing = get_summary_onecase(
    # results_dir_planes_share / losses_planes2_share.index[0]
    results_dir_planes_share/'share-plane2-724-4329'
)

Time the loss functions

In [ ]:
from functools import partial
mloss_jit = jax.jit(partial(losses.momentum_loss, forcing=forcing, datainfo=datainfo))
_ = mloss_jit(ref[:50,...])
dloss_jit = jax.jit(partial(losses.divergence, datainfo=datainfo))
_ = dloss_jit(ref[:50,...,:-1])
%timeit _ = mloss_jit(ref[:50,...]) + dloss_jit(ref[:50,...,:-1])

In [ ]:
_observed = pred_planes2_share[:50,sensors_planes2_share]
_observed_ref = ref[:50,sensors_planes2_share]
mse_jit = jax.jit(losses.mse)
_ = mse_jit(_observed, _observed_ref)
%timeit _ = mse_jit(_observed, _observed_ref)

In [ ]:
_ = mse_jit(pred_planes2_share[:50,...], ref[:50,...])
%timeit _ = mse_jit(pred_planes2_share[:50,...], ref[:50,...])

In [ ]:
pred_planes2_notshare, sensors_planes2_notshare, ref, datainfo, forcing = get_summary_onecase(
    # Path(results_dir_planes_notshare,losses_planes2_notshare.index[0])
    results_dir_planes_notshare / 'notshare-plane2-724-4329'
)

In [ ]:
# with jax.default_device(jax.devices('cpu')[0]):
vort = []
vort_planes2_share = []
vort_planes2_notshare = []
batch = 250
for i in range(ref.shape[0]//batch):
    _vort = derivatives.vorticity(ref[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort.append(np.einsum('vt... -> t...v', _vort))
    _vort_planes2_share = derivatives.vorticity(pred_planes2_share[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_planes2_share.append(np.einsum('vt... -> t...v', _vort_planes2_share))
    _vort_planes2_notshare = derivatives.vorticity(pred_planes2_notshare[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_planes2_notshare.append(np.einsum('vt... -> t...v', _vort_planes2_notshare))

vort = np.concatenate(vort, axis=0)
vort_planes2_share = np.concatenate(vort_planes2_share, axis=0)
vort_planes2_notshare = np.concatenate(vort_planes2_notshare, axis=0)

v_abs = np.sqrt(np.einsum('t...v -> t...', vort**2))
v_planes2_share_abs = np.sqrt(np.einsum('t...v -> t...', vort_planes2_share**2))
v_planes2_notshare_abs = np.sqrt(np.einsum('t...v -> t...', vort_planes2_notshare**2))

In [ ]:
v_abs_mean = v_abs.mean()
v_abs_std = v_abs.mean(axis=0).std()

In [ ]:
## Plot mean vorticity isosurface
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.3, width_ratios=[15,15,15,1], left=0.02, right=0.87, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 3*v_abs_std 

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_share_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_planes2_notshare_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\overline{\|\omega\|}$')
plt.show()
# fig.savefig(save_to_review / 'planes2-volume-vabs-mean.png')

## plot inst vorticity
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.45, width_ratios=[15,15,15,1], left=0.02, right=0.90, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_share_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_planes2_notshare_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')
fig.text(0.45,0.95,f't={datainfo.dt*plt_t_planes:.2f}')
plt.show()
# fig.savefig(save_to_review / f'planes2-volume-vabs-t{plt_t_planes}.png')


## plot inst vorticity different t
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.45, width_ratios=[15,15,15,1], left=0.02, right=0.90, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_share_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_planes2_notshare_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')
fig.text(0.45,0.95,f't={datainfo.dt*(plt_t_planes+100):.2f}')
plt.show()
# fig.savefig(save_to_review / f'planes2-volume-vabs-t{plt_t_planes+100}.png')

In [ ]:
fig9, axes = plot_observations(ref[0,...], has_values=sensors_planes2_share, figsize=(3.8,3.8), alpha=1)
# fig9.savefig(save_to_planes / 'planes2-sensors')

In [ ]:
fig10, axes = plot_compare_networks(
    [v_abs[plt_t_planes], ref[plt_t_planes,...,-1]], 
    [v_planes2_share_abs[plt_t_planes], pred_planes2_share[plt_t_planes,...,-1]],
    [v_planes2_notshare_abs[plt_t_planes], pred_planes2_notshare[plt_t_planes,...,-1]],
    labels=['$|v|$','$p$']
)
axes[2].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[5].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[3].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[3].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[4].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[4].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[5].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[5].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
# fig10.savefig(save_to_planes / 'planes2-volumes-inst.png')

In [ ]:
fig11, axes = plot_compare_networks(
    # [np.mean(ref[...,0],axis=0), np.mean(ref[...,1], axis=0)], 
    # [np.mean(pred_planes2_share[...,0],axis=0), np.mean(pred_planes2_share[...,1],axis=0)],
    # [np.mean(pred_planes2_notshare[...,0],axis=0), np.mean(pred_planes2_notshare[...,1],axis=0)],
    # labels=['$u_1$','$u_2$']
    [np.mean(ref[...,0],axis=0), np.mean(ref[...,-1], axis=0)], 
    [np.mean(pred_planes2_share[...,0],axis=0), np.mean(pred_planes2_share[...,-1],axis=0)],
    [np.mean(pred_planes2_notshare[...,0],axis=0), np.mean(pred_planes2_notshare[...,-1],axis=0)],
    labels=['$u_1$','$p$']
)
fig11.text(0.1,0.95,'Reference')
fig11.text(0.35,0.95,'Weight-sharing')
fig11.text(0.6,0.95,'PC-DualConvNet')
axes[2].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[5].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
fig11.savefig(save_to_planes / 'planes2-volumes-mean-u1-p.png')

In [ ]:
fig11, axes = plot_compare_networks(
    [np.mean(ref[...,2],axis=0), np.mean(ref[...,3], axis=0)], 
    [np.mean(pred_planes2_share[...,2],axis=0), np.mean(pred_planes2_share[...,3],axis=0)],
    [np.mean(pred_planes2_notshare[...,2],axis=0), np.mean(pred_planes2_notshare[...,3],axis=0)],
    labels=['$u_3$','$p$']
)
fig11.text(0.1,0.95,'Reference')
fig11.text(0.35,0.95,'Weight-sharing')
fig11.text(0.6,0.95,'PC-DualConvNet')
axes[2].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[5].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[3].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[3].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[4].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[4].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[5].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[5].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
fig11.savefig(save_to_planes / 'planes2-volumes-mean-u3-p')

In [ ]:
fig12, ax, spectrum_planes2 = plot_tke(ref, [pred_planes2_share, pred_planes2_notshare], ['Weight-sharing', 'PC-DualConvNet'], datainfo, log=True, linewidth=1)
# ax.set_xscale('log')
# fig12.savefig(save_to_planes / 'planes2-tke')

In [ ]:
# the observed planes are z = 16,48 and x=32
index_plot = [18,32]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_share = pred_planes2_share[plt_t_planes, :, :, index_plot,0]
vplot_pred_notshare = pred_planes2_notshare[plt_t_planes, :, :, index_plot,0]
# vplotz = v_abs[plt_t_planes, :, :, index_plot] # advance index goes to the front
# vplot_pred_share = v_planes2_share_abs[plt_t_planes, :, :, index_plot]
# vplot_pred_notshare = v_planes2_share_abs[plt_t_planes, :, :, index_plot]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_share = pred_planes2_share[plt_t_planes, :, :, index_plot, -1]
pplot_pred_notshare = pred_planes2_notshare[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig13 = plt.figure(figsize=(6.5,2.2))
gridleft = ImageGrid(fig13, (0.09,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig13, (0.56,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1, i2 = 0 + 3*j, 1 + 3*j, 2 + 3*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_share[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i2].imshow(vplot_pred_notshare[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_share[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i2].imshow(pplot_pred_notshare[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig13.text(0.005,0.55,f'$x_3={index_plot[0]*2/64:.2f}\pi$', rotation=90)
fig13.text(0.005,0.15,f'$x_3={index_plot[1]*2/64:.2f}\pi$', rotation=90)
fig13.text(0.12,0.91,'Ref.',fontsize='small')
fig13.text(0.22,0.93,'Weight-',fontsize='small')
fig13.text(0.22,0.87,'sharing',fontsize='small')
fig13.text(0.31,0.91,'PC-DualConvNet',fontsize='small')
fig13.text(0.12+0.47,0.91,'Ref.',fontsize='small')
fig13.text(0.22+0.47,0.93,'Weight-',fontsize='small')
fig13.text(0.22+0.47,0.87,'sharing',fontsize='small')
fig13.text(0.31+0.47,0.91,'PC-DualConvNet',fontsize='small')

fig13.savefig(save_to_planes / 'planes2-slice-difference-z')

In [ ]:
## Error and distance from observed plane
# the observed planes are z = 16,48 and x=32
rms_planes2_share = np.zeros((64,))
rms_planes2_notshare = np.zeros((64,))
for iz in range(64):
    rms_planes2_share[iz] = np.sqrt(losses.mse(pred_planes2_share[:,:,:,iz,:], ref[:,:,:,iz,:]))
    rms_planes2_notshare[iz] = np.sqrt(losses.mse(pred_planes2_notshare[:,:,:,iz,:], ref[:,:,:,iz,:]))

In [ ]:
fig, ax = plot_error_v_distance(rms_planes2_share, rms_planes2_notshare, measured_iz=[16,48], labels=['Weight\n-sharing','PC-DualConvNet'])
ax.set_ylim([0.08, 0.47])
ax.legend(ncols=2, loc='upper center', fontsize='small', columnspacing=1.0, framealpha=1)
plt.tight_layout()
fig.savefig('./thesis/3dkol_planes2_error_v_distance')

## One slice only

In [ ]:
pred_planes1_share, sensors_planes1_share, ref, datainfo, forcing = get_summary_onecase(
    results_dir_planes_share/'share-plane1-724-4329'
)
pred_planes1_notshare, sensors_planes1_notshare, ref, datainfo, forcing = get_summary_onecase(
    results_dir_planes_notshare / 'notshare-plane1-724-4329'
)

In [ ]:
# the observed planes are z = 32 and x=32
index_plot = [35,55]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_share = pred_planes1_share[plt_t_planes, :, :, index_plot,0]
vplot_pred_notshare = pred_planes1_notshare[plt_t_planes, :, :, index_plot,0]
# vplotz = v_abs[plt_t_planes, :, :, index_plot] # advance index goes to the front
# vplot_pred_share = v_planes2_share_abs[plt_t_planes, :, :, index_plot]
# vplot_pred_notshare = v_planes2_share_abs[plt_t_planes, :, :, index_plot]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_share = pred_planes1_share[plt_t_planes, :, :, index_plot, -1]
pplot_pred_notshare = pred_planes1_notshare[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig = plt.figure(figsize=(6.5,2.2))
gridleft = ImageGrid(fig, (0.09,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig, (0.56,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1, i2 = 0 + 3*j, 1 + 3*j, 2 + 3*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_share[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i2].imshow(vplot_pred_notshare[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_share[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i2].imshow(pplot_pred_notshare[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig.text(0.005,0.55,f'$x_3={index_plot[0]*2/64:.2f}\pi$', rotation=90)
fig.text(0.005,0.15,f'$x_3={index_plot[1]*2/64:.2f}\pi$', rotation=90)
fig.text(0.12,0.91,'Ref.',fontsize='small')
fig.text(0.22,0.93,'Weight-',fontsize='small')
fig.text(0.22,0.87,'sharing',fontsize='small')
fig.text(0.31,0.91,'PC-DualConvNet',fontsize='small')
fig.text(0.12+0.47,0.91,'Ref.',fontsize='small')
fig.text(0.22+0.47,0.93,'Weight-',fontsize='small')
fig.text(0.22+0.47,0.87,'sharing',fontsize='small')
fig.text(0.31+0.47,0.91,'PC-DualConvNet',fontsize='small')

fig.savefig(save_to_planes / 'planes1-slice-difference-z')

In [ ]:
fig, ax, spectrum_planes1 = plot_tke(ref, [pred_planes1_share, pred_planes1_notshare], ['Weight-sharing', 'PC-DualConvNet'], datainfo, log=True, linewidth=1, figsize=(3,2.2))
# ax.set_xscale('log')
# fig.savefig(save_to_planes / 'planes1-tke')

In [ ]:
fig = plt.figure(figsize=(2.5,2.2))
ax = fig.add_subplot(111, projection='3d',computed_zorder=False)
fig.subplots_adjust(left=0.005, right=0.8, top=1, bottom=0.15)
x, y, z, u = np.indices(ref[0,...].shape)
x1 = x[sensors_planes1_share]
y1 = y[sensors_planes1_share]
z1 = z[sensors_planes1_share] 
u1 = u[sensors_planes1_share]
values = ref[0,...][sensors_planes1_share]
x1, y1, z1, u1, values = x1.flatten(), y1.flatten(), z1.flatten(), u1.flatten(), values.flatten()
ax.view_init(elev=30, azim=120)
_u0 = u1 == 0
sc = ax.scatter(x1[_u0], y1[_u0], z1[_u0], c=values[_u0], marker='.', s=1, alpha=1, zorder=2)
ax.set_xlim(0.0, ref[0,...].shape[0])
ax.set_ylim(0.0, ref[0,...].shape[1])
ax.set_zlim(0.0, ref[0,...].shape[2])
ax.set_xticks([2, ref[0,...].shape[0]-2], ['0', '$2\pi$'])
ax.set_yticks([2, ref[0,...].shape[1]-2], ['0', '$2\pi$'])
ax.set_zticks([2, ref[0,...].shape[2]-2], ['0', '$2\pi$'])
ax.tick_params(pad=0.1, labelsize='x-small')
ax.set_xlabel("$x_1$",fontsize='small', labelpad=-1)
ax.set_ylabel("$x_2$",fontsize='small')
ax.set_zlabel("$x_3$",fontsize='small', labelpad=-2)
# fig.savefig(save_to_planes / 'planes1-sensors')

In [ ]:
## Error and distance from observed plane
# the observed planes are z = 32 and x=32
rms_planes1_share = np.zeros((64,))
rms_planes1_notshare = np.zeros((64,))
for iz in range(64):
    rms_planes1_share[iz] = np.sqrt(losses.mse(pred_planes1_share[:,:,:,iz,:], ref[:,:,:,iz,:]))
    rms_planes1_notshare[iz] = np.sqrt(losses.mse(pred_planes1_notshare[:,:,:,iz,:], ref[:,:,:,iz,:]))
fig, ax = plot_error_v_distance(rms_planes1_share, rms_planes1_notshare, measured_iz=[32], labels=['Weight\n-sharing','PC-DualConvNet'])
ax.set_ylim([0.08, 0.56])
ax.legend(ncols=2, loc='upper center', fontsize='small', columnspacing=1.0, framealpha=1)
plt.tight_layout()
fig.savefig('./thesis/3dkol_planes1_error_v_distance')

## Review

In [ ]:
results_dir_review = Path('../local_results/3dkol/review')

### PIV 2D2C

In [ ]:
losses_planes2_2D2C, ref = get_losses_repeats(results_dir_review, 'share-plane2-2D2C')
print(losses_planes2_2D2C.mean(), '\n', losses_planes2_2D2C.std())

In [ ]:
pred_planes2_2c_share, sensors_planes2_2c_share, ref, datainfo, forcing = get_summary_onecase(
    results_dir_review / 'share-plane2-2D2C-724-4329'
)

In [ ]:
print(losses.relative_error(pred_planes2_2c_share, ref))

In [ ]:
vort = []
vort_planes2_2d2c = []
batch = 250
for i in range(ref.shape[0]//batch):
    _vort = derivatives.vorticity(ref[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort.append(np.einsum('vt... -> t...v', _vort))
    _vort_planes2_2d2c = derivatives.vorticity(pred_planes2_2c_share[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_planes2_2d2c.append(np.einsum('vt... -> t...v', _vort_planes2_2d2c))
vort = np.concatenate(vort, axis=0)
vort_planes2_2d2c = np.concatenate(vort_planes2_2d2c, axis=0)

v_abs = np.sqrt(np.einsum('t...v -> t...', vort**2))
v_planes2_2d2c_abs = np.sqrt(np.einsum('t...v -> t...', vort_planes2_2d2c**2))

In [ ]:
v_abs_mean = v_abs.mean()
v_abs_std = v_abs.mean(axis=0).std()

In [ ]:
fig = plt.figure(figsize=(5,2.5))
gs = fig.add_gridspec(1, 3, wspace=0.45, width_ratios=[15,15,1], left=0.02, right=0.90, bottom=0.2, top=0.95)
vmin = v_abs_mean+1*v_abs_std
vmax = v_abs_mean+3*v_abs_std
ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.5, vmin=vmin, vmax=vmax)
ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_2d2c_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.5, vmax=vmax)
fig.subplots_adjust(left=0.02, right=0.87, bottom=0.2, top=0.95, wspace=0.5)
cax = fig.add_subplot(gs[2])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')
# plt.show()
fig.savefig(save_to_review / 'planes2-2d2c-vabs-mean.png')

In [ ]:
fig, axes = plot_observations(ref[0,...], has_values=sensors_planes2_2c_share, figsize=(3.8,3.8), alpha=1)
fig.savefig(save_to_review / 'planes2-2d2c-sensors.png')

In [ ]:
# the observed planes are z = 16,48 and x=32
index_plot = [18,32]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_2d2c = pred_planes2_2c_share[plt_t_planes, :, :, index_plot,0]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_2d2c = pred_planes2_2c_share[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig = plt.figure(figsize=(6,2.2))
gridleft = ImageGrid(fig, (0.1,0.15,0.35,0.75), (2,2), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig, (0.55,0.15,0.35,0.75), (2,2), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1 = 0 + 2*j, 1 + 2*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_2d2c[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_2d2c[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig.text(0.005,0.55,f'$x_3={index_plot[0]*datainfo.dz/np.pi:.2f}\pi$', rotation=90)
fig.text(0.005,0.15,f'$x_3={index_plot[1]*datainfo.dz/np.pi:.2f}\pi$', rotation=90)
fig.text(0.15,0.94,'Ref.',fontsize='small')
fig.text(0.3,0.94,'Recons.',fontsize='small')
fig.text(0.15+0.48,0.94,'Ref.',fontsize='small')
fig.text(0.3+0.46,0.94,'Recons.',fontsize='small')

# fig.savefig(save_to_review / 'planes2-2d2c-slice-difference-z')

In [ ]:
fig, ax, spectrum_planes_2d2c = plot_tke(ref, [pred_planes2_2c_share, pred_planes2_share], ['2 components','3 components'], datainfo, log=True, linewidth=1, figsize=(3,2.2))
# ax.set_xscale('log')
# fig.savefig(save_to_review / 'planes2-2d2c-tke-compare-loglin')

### Shadow in the domain

In [ ]:
shadow_origin = (25,5,34)
shadow_length = (15,15,15)

In [ ]:
losses_planes2_shadow, ref = get_losses_repeats(results_dir_review, 'share-plane2-shadow-box')
print(losses_planes2_shadow.mean(), '\n', losses_planes2_shadow.std())

In [ ]:
pred_planes2_shadow_share, sensors_planes2_shadow_share, ref, datainfo, forcing = get_summary_onecase(
    results_dir_review / 'share-plane2-shadow-box-724-4329'
)

In [ ]:
print(losses.relative_error(pred_planes2_shadow_share, ref))

In [ ]:
vort = []
vort_planes2_shadow = []
batch = 250
for i in range(ref.shape[0]//batch):
    _vort = derivatives.vorticity(ref[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort.append(np.einsum('vt... -> t...v', _vort))
    _vort_planes2_shadow = derivatives.vorticity(pred_planes2_shadow_share[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_planes2_shadow.append(np.einsum('vt... -> t...v', _vort_planes2_shadow))
vort = np.concatenate(vort, axis=0)
vort_planes2_shadow = np.concatenate(vort_planes2_shadow, axis=0)

v_abs = np.sqrt(np.einsum('t...v -> t...', vort**2))
v_planes2_shadow_abs = np.sqrt(np.einsum('t...v -> t...', vort_planes2_shadow**2))

In [ ]:
v_abs_mean = v_abs.mean()
v_abs_std = v_abs.mean(axis=0).std()

In [ ]:
index_plot = np.s_[shadow_origin[0]:shadow_origin[0]+shadow_length[0],shadow_origin[1]:shadow_origin[1]+shadow_length[1],shadow_origin[2]:shadow_origin[2]+shadow_length[2]]

fig = plt.figure(figsize=(5,2))
gs = fig.add_gridspec(1, 3, wspace=0.45, width_ratios=[10,10,1], left=0.02, right=0.90, bottom=0.2, top=0.95)
vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes,...][index_plot], lower_threshold=v_abs_mean+1*v_abs_std, s=20, sigma=0, cmap='gray', alpha=1, vmin=vmin, vmax=vmax)
ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_shadow_abs[plt_t_planes][index_plot], lower_threshold=v_abs_mean+1*v_abs_std, s=20, sigma=0, cmap='gray', alpha=1, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[2])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')

ax0.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax0.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax0.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
ax1.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax1.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax1.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
# plt.show()
fig.text(0.45,0.94,f't={datainfo.dt*plt_t_planes:.2f}')
# fig.savefig(save_to_review / f'planes2-shadow-vabs-t{plt_t_planes}-boxonly.png')


fig = plt.figure(figsize=(5,2))
gs = fig.add_gridspec(1, 3, wspace=0.45, width_ratios=[10,10,1], left=0.02, right=0.90, bottom=0.2, top=0.95)
vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes+100,...][index_plot], lower_threshold=v_abs_mean+1*v_abs_std, s=20, sigma=0, cmap='gray', alpha=1, vmin=vmin, vmax=vmax)
ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_planes2_shadow_abs[plt_t_planes+100,...][index_plot], lower_threshold=v_abs_mean+1*v_abs_std, s=20, sigma=0, cmap='gray', alpha=1, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[2])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')

ax0.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax0.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax0.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
ax1.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax1.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax1.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
# plt.show()
fig.text(0.45,0.94,f't={datainfo.dt*(plt_t_planes+100):.2f}')
# fig.savefig(save_to_review / f'planes2-shadow-vabs-t{plt_t_planes+100}-boxonly.png')

In [ ]:
fig = plt.figure(figsize=(4.5,2))
ax0 = fig.add_subplot(1,2,1,projection='3d')
ax0, scref = subplot_vort_strength(ax0, ref[...,-1].mean(axis=0)[index_plot], lower_threshold=-10, s=20, sigma=0, cmap='gray', alpha=0.6)
ax1 = fig.add_subplot(1,2,2,projection='3d')
ax1, scpred = subplot_vort_strength(ax1, pred_planes2_shadow_share[...,-1].mean(axis=0)[index_plot], lower_threshold=-10, s=20, sigma=0, cmap='gray', alpha=0.6)

ax0.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax0.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax0.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
ax1.set_xticks([2, v_abs[index_plot].shape[0]-2], [f'${shadow_origin[0]/64*2.0:.2f}\pi$', f'${(shadow_origin[0]+shadow_length[0])/64*2.0:.2f}\pi$'])
ax1.set_yticks([2, v_abs[index_plot].shape[1]-2], [f'${shadow_origin[1]/64*2.0:.2f}\pi$', f'${(shadow_origin[1]+shadow_length[1])/64*2.0:.2f}\pi$'])
ax1.set_zticks([2, v_abs[index_plot].shape[2]-2], [f'${shadow_origin[2]/64*2.0:.2f}\pi$', f'${(shadow_origin[2]+shadow_length[2])/64*2.0:.2f}\pi$'])
fig.subplots_adjust(left=0.02, right=0.87, bottom=0.2, top=0.95, wspace=0.5)
plt.show()
# fig.savefig(save_to_review / 'planes2-shadow-pressure-mean-boxonly.png')

In [ ]:
fig, axes = plot_observations(ref[0,...], has_values=sensors_planes2_shadow_share, figsize=(3.8,3.8), alpha=1)
fig.savefig(save_to_review / 'planes2-shadow-sensors.png')

In [ ]:
# the observed planes are z = 16,48 and x=32
index_plot = [34,48]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_shadow = pred_planes2_shadow_share[plt_t_planes, :, :, index_plot,0]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_shadow = pred_planes2_shadow_share[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig = plt.figure(figsize=(6,2.2))
gridleft = ImageGrid(fig, (0.1,0.15,0.35,0.75), (2,2), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig, (0.55,0.15,0.35,0.75), (2,2), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1 = 0 + 2*j, 1 + 2*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_shadow[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    # draw box
    if shadow_origin[2] <= index_plot[j] <= shadow_origin[2]+shadow_length[2]:
        shadow = patches.Rectangle(shadow_origin[:-1], shadow_length[0], shadow_length[1], edgecolor='white', facecolor='white', alpha=0.3)
        gridleft[iref].add_patch(shadow)


    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_shadow[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig.text(0.005,0.55,f'$x_3={index_plot[0]*datainfo.dz/np.pi:.2f}\pi$', rotation=90)
fig.text(0.005,0.15,f'$x_3={index_plot[1]*datainfo.dz/np.pi:.2f}\pi$', rotation=90)
fig.text(0.15,0.94,'Ref.',fontsize='small')
fig.text(0.3,0.94,'Recons.',fontsize='small')
fig.text(0.15+0.48,0.94,'Ref.',fontsize='small')
fig.text(0.3+0.46,0.94,'Recons.',fontsize='small')

# fig.savefig(save_to_review / 'planes2-shadow-slice-difference-z')

In [ ]:
fig, ax, spectrum_planes_shadow = plot_tke(ref, [pred_planes2_shadow_share, pred_planes2_share], ['Has missing area','No missing area'], datainfo, log=True, linewidth=1, spectrum=spectrum_planes_shadow, figsize=(3,2.2))
# ax.set_xscale('log')
fig.savefig(save_to_review / 'planes2-shadow-tke-compare-loglin')

## Forcing continuity

In [ ]:
divfree_dir = Path('../local_results/3dkol/sharenet-with-divfree-condition-wdiv0-849-2084')
pred_divfree, _, ref, datainfo, forcing = get_summary_onecase(divfree_dir)

In [ ]:
with jax.default_device(jax.devices("cpu")[0]):
    print(losses.momentum_loss(pred_divfree, datainfo, forcing=forcing))
    print(losses.divergence(pred_divfree[...,:-1], datainfo))
    print(losses.relative_error(pred_divfree, ref))

In [ ]:
with jax.default_device(jax.devices("cpu")[0]):
    print(losses.momentum_loss(ref, datainfo, forcing=forcing))
    print(losses.divergence(ref[...,:-1], datainfo))

In [ ]:
# the observed planes are z = [16,48] and x=32
index_plot = [18,32]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_share = pred_planes1_share[plt_t_planes, :, :, index_plot,0]
vplot_pred_divfree = pred_divfree[plt_t_planes, :, :, index_plot,0]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_share = pred_planes1_share[plt_t_planes, :, :, index_plot, -1]
pplot_pred_divfree = pred_divfree[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig = plt.figure(figsize=(6.5,2.2))
gridleft = ImageGrid(fig, (0.09,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig, (0.56,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1, i2 = 0 + 3*j, 1 + 3*j, 2 + 3*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_share[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i2].imshow(vplot_pred_divfree[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_share[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i2].imshow(pplot_pred_divfree[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig.text(0.005,0.55,f'$x_3={index_plot[0]*datainfo.dz:.2f}$', rotation=90)
fig.text(0.005,0.2,f'$x_3={index_plot[1]*datainfo.dz:.2f}$', rotation=90)
fig.text(0.12,0.91,'Ref.',fontsize='small')
fig.text(0.22,0.93,'Weight-',fontsize='small')
fig.text(0.22,0.87,'sharing',fontsize='small')
fig.text(0.32,0.93,'Divergence-',fontsize='small')
fig.text(0.35,0.87,'free',fontsize='small')
fig.text(0.12+0.47,0.91,'Ref.',fontsize='small')
fig.text(0.22+0.47,0.93,'Weight-',fontsize='small')
fig.text(0.22+0.47,0.87,'sharing',fontsize='small')
fig.text(0.32+0.47,0.93,'Divergence-',fontsize='small')
fig.text(0.35+0.47,0.87,'free',fontsize='small')

# fig.savefig(save_to_planes / 'planes2divfree-slice-difference-z')

In [ ]:
fig, ax, spectrum_planes2divfree = plot_tke(ref, [pred_planes1_share, pred_divfree], ['Weight-sharing', 'Divergence-free'], datainfo, log=True, linewidth=1, figsize=(3,2.2))
ax.set_xscale('log')
fig.savefig(save_to_planes / 'planes2divfree-tke')

# Reconstruction From Noisy data


In [ ]:
def get_losses_repeats_noisy(repeat_dir, prefix, history=False, idx_z=32):
    folders = [f for f in repeat_dir.iterdir() if f.is_dir()]
    pattern = re.compile(f'^{prefix}')
    folders = [f for f in folders if re.search(pattern, f.name)]
    print(len(folders),folders)

    l = collections.defaultdict(list)
    for i in range(len(folders)):
        print(f'    run{i}')
        if i == 0:
            pred_train, has_values, ref, datainfo, forcing, ref_noisy = get_summary_onecase(folders[i], noisy=True)
        else:
            pred_train, has_values = get_summary_onecase(folders[i], predict_only=True)

        l['rel-l2'].append(float(
            losses.relative_error(pred_train, ref)
        ))
        l['l2'].append(float(
            losses.mse(pred_train,ref)
        ))
        l['rel-l2-noisy'].append(float(
            losses.relative_error(ref_noisy, ref)
        ))
        l['l2-noisy'].append(float(
            losses.mse(ref_noisy,ref)
        ))
        observed_ref = ref[:,has_values]
        observed_pred = pred_train[:,has_values]
        l['sensor'].append(float(
            losses.mse(observed_pred, observed_ref)
        ))
        val_ref = ref[:,:,:,idx_z,:-1]
        val_pred = pred_train[:,:,:,idx_z,:-1]
        l['sensor-val'].append(float(  ## this is with clean data
            losses.mse(val_pred, val_ref)
        ))
        with jax.default_device(jax.devices('cpu')[0]):
            l['momentum'].append(float(
                losses.momentum_loss(pred_train,datainfo,forcing=forcing)
            ))
            l['div'].append(float(
                losses.divergence(pred_train[...,:-1], datainfo)
            ))
    l['rel-l2'].append(np.nan)
    l['l2'].append(np.nan)
    l['rel-l2-noisy'].append(np.nan)
    l['l2-noisy'].append(np.nan)
    l['sensor'].append(np.nan)
    l['sensor-val'].append(np.nan)
    with jax.default_device(jax.devices('cpu')[0]):
        l['momentum'].append(float(
            losses.momentum_loss(ref, datainfo, forcing=forcing)
        ))
        l['div'].append(float(
            losses.divergence(ref[...,:-1], datainfo, forcing=forcing)
        ))
    indexes = [f.name for f in folders]
    indexes.append('ref')
    df = pd.DataFrame(l, index=indexes)
    df['physics'] = df['momentum'] + df['div']
    df['total-val'] = df['physics'] + df['sensor-val']
    df.sort_values('total-val', inplace=True)
    return df, ref

In [ ]:
results_dir_noisy = Path('../local_results/3dkol/repeats_noisy')
save_to_noisy = Path('./figs-3dkol-planes/noisy/')

In [ ]:
losses_noisy_share, _ = get_losses_repeats_noisy(results_dir_noisy, 'share-')
losses_noisy_notshare, ref = get_losses_repeats_noisy(results_dir_noisy, 'notshare-')

In [ ]:
losses_noisy_share

In [ ]:
losses_noisy_notshare

In [ ]:
print(losses_noisy_share.mean(), '\n', losses_noisy_share.std())
print('')
print(losses_noisy_notshare.mean(), '\n', losses_noisy_notshare.std())

In [ ]:
pred_noisy_share, sensors_noisy_share, ref, datainfo, forcing, ref_noisy = get_summary_onecase(
    results_dir_noisy / 'share-724-4329',
    noisy=True
)
pred_noisy_notshare, sensors_noisy_notshare = get_summary_onecase(
    results_dir_noisy / 'notshare-724-4329',
    noisy=True,
    predict_only=True
)

In [ ]:
fig14, axes = plot_observations(ref_noisy[0,...], has_values=sensors_noisy_share, figsize=(3.8,3.8), alpha=1)
# fig14.savefig(save_to_noisy / 'sensors-noisy')

In [ ]:
fig15, axes = plot_compare_networks(
    [np.mean(ref[...,2],axis=0), np.mean(ref[...,3], axis=0)], 
    [np.mean(pred_noisy_share[...,2],axis=0), np.mean(pred_noisy_share[...,3],axis=0)],
    [np.mean(pred_noisy_notshare[...,2],axis=0), np.mean(pred_noisy_notshare[...,3],axis=0)],
    labels=['$u_3$', '$p$']
)
# fig15.text(0.1,0.95,'Reference')
# fig15.text(0.35,0.95,'Weight-sharing')
# fig15.text(0.6,0.95,'PC-DualConvNet')

axes[2].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[5].set_zlabel("$x_3$",fontsize='small',labelpad=-5)

axes[3].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[3].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[4].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[4].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[5].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[5].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
fig15.savefig(save_to_noisy / 'noisy-volumes-mean-u3-p.png')

In [ ]:
fig16, axes = plot_compare_networks(
    [np.mean(ref[...,0],axis=0), np.mean(ref[...,1], axis=0)], 
    [np.mean(pred_noisy_share[...,0],axis=0), np.mean(pred_noisy_share[...,1],axis=0)],
    [np.mean(pred_noisy_notshare[...,0],axis=0), np.mean(pred_noisy_notshare[...,1],axis=0)],
    labels=['$u_1$', '$u_2$']
)
fig16.text(0.1,0.95,'Reference')
fig16.text(0.35,0.95,'Weight-sharing')
fig16.text(0.6,0.95,'PC-DualConvNet')

axes[2].set_zlabel("$x_3$",fontsize='small',labelpad=-5)
axes[5].set_zlabel("$x_3$",fontsize='small',labelpad=-5)

axes[3].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[3].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[4].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[4].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
axes[5].set_xlabel("$x_1$",fontsize='small',labelpad=-5)
axes[5].set_ylabel("$x_2$",fontsize='small',labelpad=-5)
fig16.savefig(save_to_noisy / 'noisy-volumes-mean-u1-u2.png')

In [ ]:
fig17, axes, spectrum_noisy = plot_tke(
    ref,
    [pred_noisy_share, pred_noisy_notshare],
    ['Sharing weights', 'PC-Dual'],
    datainfo, 
    log=True,
    linewidth=1,
    # spectrum=spectrum_noisy
)
axes.set_xscale('log')
fig17.savefig(save_to_noisy / 'noisy-tke')

In [ ]:
# the observed planes are z = 16,48 and x=32
plt_t_noisy = 800
index_plot = [16,30]

vplotz = ref[plt_t_noisy, :, :, index_plot,0] # advance index goes to the front
vplotz_noisy = ref_noisy[plt_t_noisy, :, :, index_plot,0] # advance index goes to the front
vplot_pred_share = pred_noisy_share[plt_t_noisy, :, :, index_plot,0]
vplot_pred_notshare = pred_noisy_notshare[plt_t_noisy, :, :, index_plot,0]

pplotz = ref[plt_t_noisy, :, :, index_plot, -1]
pplotz_noisy = ref_noisy[plt_t_noisy, :, :, index_plot, -1]
pplot_pred_share = pred_noisy_share[plt_t_noisy, :, :, index_plot, -1]
pplot_pred_notshare = pred_noisy_notshare[plt_t_noisy, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig18 = plt.figure(figsize=(5,4.5))
gridleft = ImageGrid(fig18, (0.07,0.56,0.85,0.4), (2,4), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig18, (0.07,0.08,0.85,0.4), (2,4), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, inoisy, i1, i2 = 0 + 4*j, 1 + 4*j, 2 + 4*j, 3 + 4*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    imnoisy = gridleft[inoisy].imshow(vplotz_noisy[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_share[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i2].imshow(vplot_pred_notshare[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    imnoisy = gridright[inoisy].imshow(pplotz_noisy[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_share[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i2].imshow(pplot_pred_notshare[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

fig18.text(0.005,0.3,f'$x_3={index_plot[0]*2/64:.2f}\pi$', rotation=90)
fig18.text(0.005,0.08,f'$x_3={index_plot[1]*2/64:.2f}\pi$', rotation=90)
fig18.text(0.005,0.8,f'$x_3={index_plot[0]*2/64:.2f}\pi$', rotation=90)
fig18.text(0.005,0.57,f'$x_3={index_plot[1]*2/64:.2f}\pi$', rotation=90)

fig18.text(0.15,0.97,'Reference')
fig18.text(0.35,0.97,'Noisy')
fig18.text(0.45,0.97,'Weight-sharing')
fig18.text(0.67,0.97,'PC-DualConvNet')

fig18.savefig(save_to_noisy / 'noisy-slice-difference-z')

In [ ]:
## Error and distance from observed plane
# the observed planes are z = 16,48 and x=32
rms_noisy_share = np.zeros((64,))
rms_noisy_notshare = np.zeros((64,))
for iz in range(64):
    rms_noisy_share[iz] = np.sqrt(losses.mse(pred_noisy_share[:,:,:,iz,:], ref[:,:,:,iz,:]))
    rms_noisy_notshare[iz] = np.sqrt(losses.mse(pred_noisy_notshare[:,:,:,iz,:], ref[:,:,:,iz,:]))
fig, ax = plot_error_v_distance(rms_noisy_share, rms_noisy_notshare, measured_iz=[16,48], labels=['Weight\n-sharing','PC-DualConvNet'])
ax.set_ylim([0.12, 0.45])
ax.legend(ncols=2, loc='upper center', fontsize='small', columnspacing=1.0, framealpha=1)
plt.tight_layout()
fig.savefig('./thesis/3dkol_noisy_error_v_distance')

In [ ]:
# with jax.default_device(jax.devices('cpu')[0]):
vort = []
vort_noisy_share = []
vort_noisy_notshare = []
batch = 250
for i in range(ref.shape[0]//batch):
    _vort = derivatives.vorticity(ref[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort.append(np.einsum('vt... -> t...v', _vort))
    _vort_noisy_share = derivatives.vorticity(pred_noisy_share[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_noisy_share.append(np.einsum('vt... -> t...v', _vort_noisy_share))
    _vort_noisy_notshare = derivatives.vorticity(pred_noisy_notshare[i*batch:(i+1)*batch,...,:-1], datainfo)
    vort_noisy_notshare.append(np.einsum('vt... -> t...v', _vort_noisy_notshare))

vort = np.concatenate(vort, axis=0)
vort_noisy_share = np.concatenate(vort_noisy_share, axis=0)
vort_noisy_notshare = np.concatenate(vort_noisy_notshare, axis=0)

v_abs = np.sqrt(np.einsum('t...v -> t...', vort**2))
v_noisy_share_abs = np.sqrt(np.einsum('t...v -> t...', vort_noisy_share**2))
v_noisy_notshare_abs = np.sqrt(np.einsum('t...v -> t...', vort_noisy_notshare**2))

In [ ]:
v_abs_mean = v_abs.mean()
v_abs_std = v_abs.mean(axis=0).std()

In [ ]:
## Plot mean vorticity isosurface
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.3, width_ratios=[15,15,15,1], left=0.02, right=0.87, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 3*v_abs_std 

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_noisy_share_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_noisy_notshare_abs.mean(axis=0), lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\overline{\|\omega\|}$')
plt.show()
# fig.savefig(save_to_review / 'noisy-volume-vabs-mean.png')

## plot inst vorticity
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.45, width_ratios=[15,15,15,1], left=0.02, right=0.90, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_noisy_share_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_noisy_notshare_abs[plt_t_planes,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')
fig.text(0.45,0.95,f't={datainfo.dt*plt_t_planes:.2f}')
plt.show()
fig.savefig(save_to_review / f'noisy-volume-vabs-t{plt_t_planes}.png')


## plot inst vorticity different t
fig = plt.figure(figsize=(7.5,2.5))
gs = fig.add_gridspec(1, 4, wspace=0.45, width_ratios=[15,15,15,1], left=0.02, right=0.90, bottom=0.2, top=0.95)

vmin, vmax = v_abs_mean + 1*v_abs_std, v_abs_mean + 8*v_abs_std

ax0 = fig.add_subplot(gs[0],projection='3d')
ax0, scref = subplot_vort_strength(ax0, v_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax1 = fig.add_subplot(gs[1],projection='3d')
ax1, scpred = subplot_vort_strength(ax1, v_noisy_share_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

ax2 = fig.add_subplot(gs[2],projection='3d')
ax2, scpred = subplot_vort_strength(ax2, v_noisy_notshare_abs[plt_t_planes+100,...], lower_threshold=vmin, s=5, sigma=0, cmap='gray', alpha=0.7, vmin=vmin, vmax=vmax)

cax = fig.add_subplot(gs[3])
cbar = fig.colorbar(scref, cax=cax, label='$\|\omega\|$')
fig.text(0.45,0.95,f't={datainfo.dt*(plt_t_planes+100):.2f}')
plt.show()
fig.savefig(save_to_review / f'noisy-volume-vabs-t{plt_t_planes+100}.png')

Noisy divfree network

In [ ]:
divfree_noisy_dir = Path('../local_results/3dkol/noisy-sharenet-with-divfree-condition-wdiv0-724-4329')
pred_noisy_divfree, _, ref, datainfo, forcing, ref_noisy = get_summary_onecase(divfree_noisy_dir, noisy=True)

In [ ]:
with jax.default_device(jax.devices("cpu")[0]):
    print(losses.momentum_loss(pred_noisy_divfree, datainfo, forcing=forcing))
    print(losses.divergence(pred_noisy_divfree[...,:-1], datainfo))
    print(losses.relative_error(pred_noisy_divfree, ref))

In [ ]:
# the observed planes are z = [16,48] and x=32
index_plot = [18,25]

vplotz = ref[plt_t_planes, :, :, index_plot,0] # advance index goes to the front
vplot_pred_share = pred_noisy_share[plt_t_planes, :, :, index_plot,0]
vplot_pred_noisy_divfree = pred_noisy_divfree[plt_t_planes, :, :, index_plot,0]

pplotz = ref[plt_t_planes, :, :, index_plot, -1]
pplot_pred_share = pred_noisy_share[plt_t_planes, :, :, index_plot, -1]
pplot_pred_noisy_divfree = pred_noisy_divfree[plt_t_planes, :, :, index_plot, -1]
vmax = vplotz.max()
vmin = vplotz.min()
pmax = pplotz.max()
pmin = pplotz.min()

fig = plt.figure(figsize=(6.5,2.2))
gridleft = ImageGrid(fig, (0.09,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)
gridright= ImageGrid(fig, (0.56,0.1,0.35,0.85), (2,3), cbar_mode='single', share_all=True)

for j in range(len(index_plot)):
    iref, i1, i2 = 0 + 3*j, 1 + 3*j, 2 + 3*j

    imref = gridleft[iref].imshow(vplotz[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i1].imshow(vplot_pred_share[j,...].T, vmax=vmax, vmin=vmin)
    gridleft[i2].imshow(vplot_pred_noisy_divfree[j,...].T, vmax=vmax, vmin=vmin)
    cbar = gridleft.cbar_axes[0].colorbar(imref)
    cbar.set_label('$u_1$', labelpad=-5)
    
    # pressure
    imref = gridright[iref].imshow(pplotz[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i1].imshow(pplot_pred_share[j,...].T, vmax=pmax, vmin=pmin)
    gridright[i2].imshow(pplot_pred_noisy_divfree[j,...].T, vmax=pmax, vmin=pmin)
    cbar = gridright.cbar_axes[0].colorbar(imref)
    cbar.set_label('$p$', labelpad=-1)

for ax in gridleft.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')
for ax in gridright.axes_all:
    ax.set_xlabel('$x_1$',labelpad=0.0)
    ax.set_ylabel('$x_2$', labelpad=0.0)
    ax.set(yticks=[3,60], yticklabels=['0', '$2\pi$'], xticks=[4,57], xticklabels=['0', '$2\pi$'])
    ax.tick_params(length=0, labelsize='x-small')

# _z1 = f'{}' 
# _z1 = f'{index_plot[0]*datainfo.dz:.2f}' 
fig.text(0.005,0.55,f'$x_3={index_plot[0]*datainfo.dz:.2f}$', rotation=90)
fig.text(0.005,0.2,f'$x_3={index_plot[1]*datainfo.dz:.2f}$', rotation=90)
fig.text(0.12,0.91,'Ref.',fontsize='small')
fig.text(0.22,0.93,'Weight-',fontsize='small')
fig.text(0.22,0.87,'sharing',fontsize='small')
fig.text(0.32,0.93,'Divergence-',fontsize='small')
fig.text(0.35,0.87,'free',fontsize='small')
fig.text(0.12+0.47,0.91,'Ref.',fontsize='small')
fig.text(0.22+0.47,0.93,'Weight-',fontsize='small')
fig.text(0.22+0.47,0.87,'sharing',fontsize='small')
fig.text(0.32+0.47,0.93,'Divergence-',fontsize='small')
fig.text(0.35+0.47,0.87,'free',fontsize='small')

fig.savefig(save_to_noisy / 'divfree-slice-difference-z')

In [ ]:
fig, ax, spectrum_planes2divfree = plot_tke(ref, [pred_noisy_share, pred_noisy_divfree], ['Weight-sharing', 'Divergence-free'], datainfo, log=True, linewidth=1, figsize=(3,2.2), spectrum=spectrum_planes2divfree)
ax.set_xscale('log')
fig.savefig(save_to_noisy / 'divfree-tke')

# Plot data

In [ ]:
fdata_in_thesis = [
        Path('/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_478_t200-250.h5'),
        Path('/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_478_t360-410.h5'),
        Path('/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_1372_t200-250.h5'),
        Path('/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_57_t180-230.h5'),
]
data_dict = get_data_convergence(*fdata_in_thesis)

In [ ]:
fig8, axes = plot_data_convergence(data_dict, figsize=(7,4))

In [ ]:
# axes['tke'].set_xscale('linear')
fig8.text(0.08,0.06,'$x_1$',fontsize='small',rotation=345)
fig8.text(0.38,0.06,'$x_1$',fontsize='small',rotation=345)
fig8.text(0.08,0.53,'$x_1$',fontsize='small',rotation=345)
fig8.text(0.38,0.53,'$x_1$',fontsize='small',rotation=345)
fig8.text(0.22,0.07,'$x_2$',fontsize='small',rotation=55)
fig8.text(0.52,0.07,'$x_2$',fontsize='small',rotation=55)
fig8.text(0.22,0.55,'$x_2$',fontsize='small',rotation=55)
fig8.text(0.52,0.55,'$x_2$',fontsize='small',rotation=55)
fig8.text(0.26,0.26,'$x_3$',fontsize='small')
fig8.text(0.56,0.26,'$x_3$',fontsize='small')
fig8.text(0.26,0.74,'$x_3$',fontsize='small')
fig8.text(0.56,0.74,'$x_3$',fontsize='small')
fig8.savefig(save_to_2dmethod + 'data')

# Noisy data hyperparameters

In [ ]:
sweeplosses_share = pd.read_csv('../local_results/3dkol/sweep_share_noisy/SweepLosses.csv', index_col=0)
sweeplosses_notshare = pd.read_csv('../local_results/3dkol/sweep_noisy_notshare/SweepLosses.csv', index_col=0)

In [ ]:
sweeplosses_share['diff-sensor'] = np.abs(sweeplosses_share['sensor-train-pred'] - sweeplosses_share['sensor-val-pred'])
sweeplosses_notshare['diff-sensor'] = np.abs(sweeplosses_notshare['sensor-train-pred'] - sweeplosses_notshare['sensor-val-pred'])
sweeplosses_share['physics-pred'] = np.abs(sweeplosses_share['momentum-pred'] + sweeplosses_share['div-pred'])
sweeplosses_notshare['physics-pred'] = np.abs(sweeplosses_notshare['momentum-pred'] + sweeplosses_notshare['div-pred'])
sweeplosses_share['rel-l2-pred'] = sweeplosses_share['rel-l2-pred']*100
sweeplosses_notshare['rel-l2-pred'] = sweeplosses_notshare['rel-l2-pred']*100

In [ ]:
data_std = 0.47131678

In [ ]:
data_utils.signal_noise_ratio(data_std**2, sweeplosses_share['sensor-val-pred'])

In [ ]:
sweeplosses_share.loc['laced-sweep-12']['physics-pred']

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(5,4), height_ratios=[2,1])
sc = sweeplosses_share.plot.scatter('physics-pred', 'sensor-val-pred', c='rel-l2-pred', ax=axes[0], cmap=my_continuous_cmap)
sc.collections[0].colorbar.set_label("Relative error")
# # Annotate each point
# for k,v in sweeplosses_share.iterrows():
#     axes[0].annotate(k, (v['physics-pred'], v['sensor-val-pred']))
axes[0].set_yscale('log')
axes[0].set(ylabel='Sensor loss \nvalidation plane', xlabel='Physics loss')
# this is the one I chose
_chosen = 'laced-sweep-12'
axes[0].annotate(
    '*', 
    (sweeplosses_share.loc[_chosen]['physics-pred'], sweeplosses_share.loc[_chosen]['sensor-val-pred']-0.0001),
    color='red'
)



sweeplosses_share.plot.scatter('sensor-train-pred', 'sensor-val-pred', ax=axes[1], color=my_discrete_cmap(0))
axes[1].set(ylabel='Sensor loss \nvalidation plane', xlabel='Sensor loss training planes')
fig.tight_layout()
fig.savefig(save_to_noisy / 'params_share')
# plt.show()

sweeplosses_share.sort_values('rel-l2-pred').head(5)

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(5,4), height_ratios=[2,1])
sc = sweeplosses_notshare.plot.scatter('physics-pred', 'sensor-val-pred', c='rel-l2-pred', ax=axes[0], cmap=my_continuous_cmap)
sc.collections[0].colorbar.set_label("Relative error")
# # Annotate each point
# for k,v in sweeplosses_notshare.iterrows():
#     axes[0].annotate(k, (v['physics-pred'], v['sensor-val-pred']))
# this is the chosen one
_chosen = 'whole-sweep-2'
axes[0].annotate(
    '*', 
    (sweeplosses_notshare.loc[_chosen]['physics-pred'], sweeplosses_notshare.loc[_chosen]['sensor-val-pred']-0.003),
    color='red'
)
axes[0].set_yscale('log')
axes[0].set(ylabel='Sensor loss \nvalidation plane', xlabel='Physics loss')
sweeplosses_notshare.plot.scatter('sensor-train-pred', 'sensor-val-pred', ax=axes[1], color=my_discrete_cmap(0))
axes[1].set(ylabel='Sensor loss \nvalidation plane', xlabel='Sensor loss training planes')
fig.tight_layout()
fig.savefig(save_to_noisy / 'params_notshare')
# plt.show()
sweeplosses_notshare.sort_values('rel-l2-pred').head(5)

# Clean data hyperparameters

## Hyperparameters

In [ ]:
testmin_share_sweep_dir = Path("../local_results/3dkol/sweep_share_8planes")
testmin_notshare_sweep_dir = Path("../local_results/3dkol/sweep_notshare_8planes")
wandb_log_frequency = 10

In [ ]:
sweep_summary_share = pd.read_csv(testmin_share_sweep_dir / "sweep_total_loss_share_8planes.csv")
sweep_summary_notshare = pd.read_csv(testmin_notshare_sweep_dir / "sweep_total_loss_notshare_8planes.csv")

In [ ]:
sweep_summary_notshare['Step'] = sweep_summary_notshare['Step']*wandb_log_frequency
sweep_summary_share['Step'] = sweep_summary_share['Step']*wandb_log_frequency

In [ ]:
best_run_notshare = "summer-sweep-1"
best_run_share = "efficient-sweep-23"

In [ ]:
clabels_notshare = sweep_summary_notshare.columns.str.endswith('loss_total')
plot_summary_notshare = sweep_summary_notshare.loc[:,clabels_notshare]
plot_summary_notshare.columns = [a[0] for a in plot_summary_notshare.columns.str.split(" ")]
clabels_share = sweep_summary_share.columns.str.endswith('loss_total')
plot_summary_share = sweep_summary_share.loc[:,clabels_share]
plot_summary_share.columns = [a[0] for a in plot_summary_share.columns.str.split(" ")]

fig19, axes = plt.subplots(1,2,figsize=(6.2,3))
axes[0].loglog(sweep_summary_notshare['Step'], plot_summary_notshare.loc[:,plot_summary_notshare.columns != best_run_notshare], color='k', linewidth=0.5)
axes[0].loglog(sweep_summary_notshare['Step'], plot_summary_notshare.loc[:,best_run_notshare], color='r', linewidth=0.5)
axes[1].loglog(sweep_summary_share['Step'], plot_summary_share.loc[:,plot_summary_share.columns != best_run_share], color='k', linewidth=0.5)
axes[1].loglog(sweep_summary_share['Step'], plot_summary_share.loc[:,best_run_share], color='r', linewidth=0.5)
fig19.subplots_adjust(wspace=0.4,top=0.98, bottom=0.2, right=0.98)
axes[0].set(xlabel='Epoch',ylabel='$\mathcal{L}_o + \mathcal{L}_p$')
axes[1].set(xlabel='Epoch',ylabel='$\mathcal{L}_o + \mathcal{L}_p$')
plt.show()
# fig19.savefig(save_to_planes / 'sweep-loss-total')

## Minimum number of planes

In [ ]:
testmin8_dir_share = Path(testmin_share_sweep_dir, best_run_share)
testmin8_dir_notshare = Path(testmin_notshare_sweep_dir, best_run_notshare)
testmin_repeats_share = Path("../local_results/3dkol/repeats_planes_share/")
testmin_repeats_notshare = Path("../local_results/3dkol/repeats_planes_notshare/")

In [ ]:
min_plane_share_summary = pd.read_csv(testmin_repeats_share / 'summary_3dkol_find_minimum_planes_share.csv', index_col=0)
min_plane_notshare_summary = pd.read_csv(testmin_repeats_notshare / 'summary_3dkol_find_minimum_planes_notshare.csv', index_col=0)

In [ ]:
min_plane_share_mean = min_plane_share_summary.groupby('num_planes').mean()
min_plane_notshare_mean = min_plane_notshare_summary.groupby('num_planes').mean()
print(min_plane_share_mean)
print(min_plane_notshare_mean)

In [ ]:
fig20, axes = plt.subplots(1,2,figsize=(6.2,2.5),sharex=True)
axes[0].set_xticks([1,2,4,8])
labels = ['PC-DualConvNet', 'Weight-sharing']
for i,_df in enumerate([min_plane_notshare_mean, min_plane_share_mean]):
    # axes[0].scatter(_df.index,_df['sensor_loss'], color=cmap_trafficlight(i), marker='s', zorder=i)
    axes[0].scatter(_df.index,_df['mse'], color=my_discrete_cmap(i), marker='s', zorder=i+2, label=labels[i])
    axes[1].scatter(_df.index,_df['physics_loss'], color=my_discrete_cmap(i), marker='s')
axes[0].set(xlabel='Number of $x_1-x_2$ planes', ylabel='MSE')
axes[1].set(xlabel='Number of $x_1-x_2$ planes', ylabel='Physics loss')
_handles, _labels = axes[0].get_legend_handles_labels()
fig20.subplots_adjust(top=0.8, wspace=0.4, bottom=0.2, right=0.95)
fig20.legend(handles=_handles, ncol=2, loc='upper center', bbox_to_anchor=(0.5,1.0))
# fig20.savefig(save_to_planes / 'min-planes-mse-and-physics')

fig21, ax = plt.subplots(1,1,figsize=(4,2.5))
for i,_df in enumerate([min_plane_notshare_mean, min_plane_share_mean]):
    ax.scatter(_df.index,_df['sensor_loss']+_df['physics_loss'], color=my_discrete_cmap(i), marker='s')